## imports

In [1]:
import os
import warnings
import logging
from typing import Tuple, Dict
from pypdf import PdfReader
from transformers import pipeline
import torch

# Suppress pypdf warnings about wrong pointing objects
logging.getLogger('pypdf').setLevel(logging.ERROR)

# model setup 

In [2]:
pipe = pipeline("text-generation", model="google/gemma-3-1b-it", device="cpu", dtype=torch.bfloat16)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

# input the two documents

## function

In [3]:
def input_documents(pdf1_path: str, pdf2_path: str) -> Tuple[str, str]:
    """
    Load and validate two PDF documents.
    
    Args:
        pdf1_path: Path to the first PDF document
        pdf2_path: Path to the second PDF document
    
    Returns:
        Tuple containing the text content from both PDFs
    
    Raises:
        FileNotFoundError: If either PDF file does not exist
        ValueError: If either file is not a valid PDF or cannot be read
    """
    # Validate that both files exist
    if not os.path.exists(pdf1_path):
        raise FileNotFoundError(f"First PDF not found: {pdf1_path}")
    if not os.path.exists(pdf2_path):
        raise FileNotFoundError(f"Second PDF not found: {pdf2_path}")
    
    # Validate file extensions
    if not pdf1_path.lower().endswith('.pdf'):
        raise ValueError(f"First file is not a PDF: {pdf1_path}")
    if not pdf2_path.lower().endswith('.pdf'):
        raise ValueError(f"Second file is not a PDF: {pdf2_path}")
    
    # Extract text from first PDF
    try:
        reader1 = PdfReader(pdf1_path)
        text1 = ""
        for page in reader1.pages:
            text1 += page.extract_text()
        
        if not text1.strip():
            raise ValueError(f"First PDF contains no extractable text: {pdf1_path}")
    except Exception as e:
        raise ValueError(f"Failed to read first PDF {pdf1_path}: {str(e)}")
    
    # Extract text from second PDF
    try:
        reader2 = PdfReader(pdf2_path)
        text2 = ""
        for page in reader2.pages:
            text2 += page.extract_text()
        
        if not text2.strip():
            raise ValueError(f"Second PDF contains no extractable text: {pdf2_path}")
    except Exception as e:
        raise ValueError(f"Failed to read second PDF {pdf2_path}: {str(e)}")
    
    return text1, text2

## Test all nine input combinations

In [4]:
# Define the nine input combinations
test_combinations = [
    ("cis-r1.pdf", "cis-r1.pdf"),
    ("cis-r1.pdf", "cis-r2.pdf"),
    ("cis-r1.pdf", "cis-r3.pdf"),
    ("cis-r1.pdf", "cis-r4.pdf"),
    ("cis-r2.pdf", "cis-r2.pdf"),
    ("cis-r2.pdf", "cis-r3.pdf"),
    ("cis-r2.pdf", "cis-r4.pdf"),
    ("cis-r3.pdf", "cis-r3.pdf"),
    ("cis-r3.pdf", "cis-r4.pdf"),
]
PDFoutputs = [[],[],[],[],[],[],[],[],[]]

pdf_dir = "PDFS"

# Test each combination
for i, (pdf1, pdf2) in enumerate(test_combinations, 1):
    print(f"\nInput-{i}: {pdf1} and {pdf2}")
    try:
        pdf1_path = os.path.join(pdf_dir, pdf1)
        pdf2_path = os.path.join(pdf_dir, pdf2)
        doc1, doc2 = input_documents(pdf1_path, pdf2_path)
        PDFoutputs[i-1].append(doc1) 
        PDFoutputs[i-1].append(doc2)
        print(f"  ✓ Successfully loaded both documents")
        print(f"  - First PDF: {len(doc1)} characters")
        print(f"  - Second PDF: {len(doc1)} characters")
        print(f"  - First 100 chars of PDF 1: {doc1[:100].strip()}...")
    except Exception as e:
        print(f"  ✗ Error: {str(e)}")


Input-1: cis-r1.pdf and cis-r1.pdf
  ✓ Successfully loaded both documents
  - First PDF: 208021 characters
  - Second PDF: 208021 characters
  - First 100 chars of PDF 1: CIS Amazon Elastic Kubernetes Service (EKS) Benchmark v1.2.0 - 11-23-2022   Page 1 
Terms of Use Ple...

Input-2: cis-r1.pdf and cis-r2.pdf
  ✓ Successfully loaded both documents
  - First PDF: 208021 characters
  - Second PDF: 208021 characters
  - First 100 chars of PDF 1: CIS Amazon Elastic Kubernetes Service (EKS) Benchmark v1.2.0 - 11-23-2022   Page 1 
Terms of Use Ple...

Input-3: cis-r1.pdf and cis-r3.pdf
  ✓ Successfully loaded both documents
  - First PDF: 208021 characters
  - Second PDF: 208021 characters
  - First 100 chars of PDF 1: CIS Amazon Elastic Kubernetes Service (EKS) Benchmark v1.2.0 - 11-23-2022   Page 1 
Terms of Use Ple...

Input-4: cis-r1.pdf and cis-r4.pdf
  ✓ Successfully loaded both documents
  - First PDF: 208021 characters
  - Second PDF: 208021 characters
  - First 100 chars of PDF 1: 

# prompts functions

## zero shot WIP

In [5]:
def zero_shot_prompt(doc1: str, doc2: str) -> str:
    """
    Construct a zero-shot prompt to identify key data elements in two input documents.
    
    Args:
        doc1: Text content of the first document
        doc2: Text content of the second document
    
    Returns:
        A string containing the zero-shot prompt for the LLM
    """
    prompt = f"""You are an expert document analyst. Your task is to analyze two documents and identify key data elements present in each.

Document 1:
{doc1}

Document 2:
{doc2}

Please analyze both documents and identify:
1. Key data elements (e.g., version numbers, dates, entity names, specifications)
2. Common elements that appear in both documents
3. Elements that are unique to each document
4. Any significant differences in how similar information is presented

Provide your analysis in a structured format."""
    
    return prompt

# prompts use

## zero shot

In [ ]:
zeroShotOutputs = []
for item in PDFoutputs:
   promptoutput = zero_shot_prompt(item[0], item[1])
   print(promptoutput)
   zeroShotOutputs.append(promptoutput)

You are an expert document analyst. Your task is to analyze two documents and identify key data elements present in each.

Document 1:
CIS Amazon Elastic Kubernetes Service (EKS) Benchmark v1.2.0 - 11-23-2022   Page 1 
Terms of Use Please see the below link for our current terms of use: https://www.cisecurity.org/cis-securesuite/cis-securesuite-membership-terms-of-use/ Page 2 
Table of Contents Terms of Use .................................................................................................................. 1 Table of Contents ........................................................................................................... 2 Overview ......................................................................................................................... 5 Intended Audience ................................................................................................................... 5 Consensus Guidance ........................................................